# Forecasting Monthly Auto-Insurance Claim Counts with PROC FORECAST

## Executive Summary

An actuarial reserving team needs a 12-month-ahead view of monthly auto-insurance claim counts that show steady portfolio growth plus a pronounced winter-weather lift. This notebook generates five years of synthetic monthly claim counts (60 months, Jan 2020 - Dec 2024, ranging from about 1,460 to 2,780 claims), then uses **PROC FORECAST** to fit a stepwise-autoregressive baseline and a multiplicative Holt-Winters seasonal model. Each model produces 12 monthly point forecasts with 95% confidence limits for capacity and reserve planning. The seasonal Holt-Winters model projects the strongest demand one-to-two months ahead (about 2,780-2,900 claims) easing to a trough around step nine (about 2,200), while the autoregressive baseline projects a smoother decay; both confidence bands widen with the horizon.

## Data Sources

| Dataset | Rows | Grain | Key Variables | Description |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | One row per calendar month (Jan 2020 - Dec 2024) | `date` (SAS date, `MONYY7.`), `claim_count` (numeric) | Synthetic monthly auto-insurance claim counts built from a linear growth trend (~9 claims/month), a sinusoidal annual cycle, an additive Dec/Jan/Feb winter-weather surge, and Gaussian noise (`rand('normal')`). Seed `20240531` makes it reproducible. |

# Forecasting Monthly Auto-Insurance Claim Counts

Reserving and capacity-planning teams at a personal-lines insurer need a forward look at how many **auto claims** will be reported each month. Claim volume in this book grows steadily as the portfolio expands, and it spikes every winter when ice, snow, and reduced daylight drive up collision and glass claims.

This notebook walks through a complete `PROC FORECAST` workflow:

1. Generate a realistic, fully synthetic monthly claim-count series.
2. Fit a **stepwise-autoregressive (STEPAR)** baseline that captures trend plus autocorrelation.
3. Fit a **multiplicative Holt-Winters (WINTERS)** model that explicitly models the 12-month seasonal cycle.
4. Compare the two models and interpret the forward forecast and confidence band.

Everything runs on inline synthetic data — no external files or network access.

## Step 1 — Generate the synthetic claims series

The DATA step below builds **60 monthly observations** (Jan 2020 through Dec 2024). For each month we combine four components that mirror a real auto book:

- **Trend** — a baseline of ~1,800 claims growing by ~9 per month as exposures rise.
- **Annual cycle** — a sine/cosine term giving a smooth seasonal wave.
- **Winter surge** — an additive bump in December, January, and February.
- **Noise** — `rand('normal', 0, 90)` for realistic month-to-month variability.

`call streaminit()` fixes the random stream so the notebook is reproducible. The `date` variable is a true SAS date built with `INTNX` and formatted `MONYY7.`, and the `ID date INTERVAL=MONTH` statement names it as the time identifier. The first 14 rows printed below land between roughly 1,460 and 2,450 claims.

In [1]:
data claims;
    call streaminit(20240531);
    do i = 0 to 59;
        /* Build a real SAS month date so INTERVAL=MONTH aligns */
        date = intnx('month', '01JAN2020'd, i);
        format date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = Jan ... 12 = Dec */

        trend  = 1800 + 9 * i;               /* growing exposure base   */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx in (12, 1, 2));   /* ice/snow surge  */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        if claim_count < 0 then claim_count = 0;
        output;
    end;
    keep date claim_count;
run;

proc print data=claims(obs=14) label;
    label date = 'Month' claim_count = 'Reported Claims';
    title 'First 14 Months of Synthetic Auto Claim Counts';
run;

                                     First 14 Months of Synthetic Auto Claim Counts                                     

  Obs  Month  Reported Claims
    1  21915             2178
    2  21946             2281
    3  21975             2252
    4  22006             1974
    5  22036             2067
    6  22067             1805
    7  22097             1697
    8  22128             1619
    9  22159             1462
   10  22189             1688
   11  22220             1713
   12  22250             2008
   13  22281             2116
   14  22312             2451

... 46 more observations (showing 14 of 60)

NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Step 2 — Stepwise-autoregressive baseline (METHOD=STEPAR)

`METHOD=STEPAR` is the default. It first fits a time-trend model — here `TREND=2` for a linear trend — then applies a **stepwise autoregression** to the residuals, entering and retaining AR lags by significance. `AR=4` caps the candidate autoregressive order at four lags, which is plenty for a monthly series with short-run momentum.

Key options used here:

- `LEAD=12` — forecast 12 months beyond the data.
- `ALPHA=0.05` — 95% confidence limits.
- `OUTFULL` — stack the historical `ACTUAL` rows together with the forecast-horizon rows in the `OUT=` dataset (rather than the forecast rows alone).
- `OUTLIMIT` — add the lower/upper confidence-limit **columns** (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` — names the time identifier and declares the series is monthly.

In [2]:
proc forecast data=claims
        out=fc_stepar
        method=stepar trend=2 ar=4
        lead=12 alpha=0.05
        outfull outlimit;
    id date interval=month;
    var claim_count;
run;

proc print data=fc_stepar(obs=10) label;
    title 'STEPAR Output: Actual, Fitted, and Forecast Rows';
run;

                                     First 14 Months of Synthetic Auto Claim Counts                                     

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                    STEPAR Output: Actual, Fitted, and Forecast Rows                                    

  Obs   DATE  _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT
    1  21915  ACTUAL  2121.816667
    2  21946  ACTUAL  2167.711419
    3  21975  ACTUAL  2254.781177
    4  22006  ACTUAL  2228.505912
    5  22036  ACTUAL  1978.152296
    6  22067  ACTUAL  2030.606563
    7  22097  ACTUAL  1864.520418
    8  22128  ACTUAL  1784.855682
    9  22159  ACTUAL  1740.781553
   10  22189  ACTUAL  1657.132366

... 62 more observations (showing 10 of 72)

NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 ob

### Reading the OUT= dataset

The `OUT=` dataset stacks rows by a `_TYPE_` column and carries the confidence limits as side **columns**:

| Element | Kind | Meaning |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | row | The observed `claim_count` for each of the 60 historical months. |
| `_TYPE_ = 'FORECAST'` | row | The 12 point predictions for the forecast horizon. |
| `L95_claim_count` / `U95_claim_count` | column | Lower / upper 95% confidence limits, populated on the `FORECAST` rows (missing on `ACTUAL` rows). The numeric level reflects `ALPHA=`. |

So the printed `OUT=` here holds 72 rows: 60 `ACTUAL` history rows followed by 12 `FORECAST` horizon rows. The first ten rows shown above are all `ACTUAL`, with the confidence-limit columns missing because limits attach only to the forecast rows.

> **Engine note.** SAS `OUTFULL` also writes a within-sample one-step-ahead `FORECAST` row for each historical month, and `OUTRESID` adds `_TYPE_='RESIDUAL'` rows. Jenner's PROC FORECAST does not yet emit those in-sample fitted/residual rows (tracked as gap test `400979`), so this notebook reads the `ACTUAL` history and the forward `FORECAST` horizon only.

## Step 3 — Seasonal Holt-Winters model (METHOD=WINTERS)

The STEPAR model captures trend and short-run autocorrelation but does not explicitly model the recurring December–February lift. For a series with a clear annual rhythm, **multiplicative Holt-Winters** is the better tool.

`METHOD=WINTERS` decomposes the series into level, linear trend (`TREND=2`), and a **multiplicative seasonal factor**. `SEASONS=12` declares a 12-period (monthly) seasonal cycle. We again request a 12-month `LEAD` with 95% limits and `OUTFULL OUTLIMIT` so the output lines up with the STEPAR run.

Because the engine advances the forecast `ID` by one unit per step (rather than honoring `INTERVAL=MONTH` for the horizon dates — gap test `400979`), the cell below reviews the forecast by **months ahead (step 1-12)** instead of relying on the printed calendar dates.

In [3]:
proc forecast data=claims
        out=fc_winters
        method=winters seasons=12 trend=2
        lead=12 alpha=0.05
        outfull outlimit;
    id date interval=month;
    var claim_count;
run;

/* Keep the 12-month forward horizon and index it by step (1..12). */
data horizon;
    set fc_winters;
    retain months_ahead 0;
    if _type_ = 'FORECAST';
    months_ahead + 1;
    keep months_ahead claim_count l95_claim_count u95_claim_count;
run;

proc print data=horizon label;
    label months_ahead     = 'Months Ahead'
          claim_count       = 'Forecast Claims'
          l95_claim_count   = 'Lower 95%'
          u95_claim_count   = 'Upper 95%';
    title 'Holt-Winters 12-Month Forward Forecast (by step)';
run;

                                    STEPAR Output: Actual, Fitted, and Forecast Rows                                    

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                    Holt-Winters 12-Month Forward Forecast (by step)                                    

  Obs  Months Ahead  Forecast Claims    Lower 95%    Upper 95%
    1             1       2783.07951  2577.844742  2988.314278
    2             2      2897.355557  2607.109764  3187.601349
    3             3      2805.272075  2449.795029   3160.74912
    4             4      2664.498049  2254.028514  3074.967585
    5             5      2628.810136  2169.891244  3087.729029
    6             6       2548.39319  2045.672732  3051.113649
    7             7      2391.649756    1848.6496  2934.649912
    8             8      2239.948352  1659.456768  2820.439936
    9             9      2197.109273  1581.404969  2812.813576
   10            

## Step 4 — Compare the two models head to head

The clearest way to judge whether the seasonal model is earning its keep is to lay its 12-month forecast next to the STEPAR baseline, step by step. We pull the `FORECAST` rows from both `OUT=` datasets, index each by months-ahead, and merge them so the divergence is visible at a glance.

The two methods agree on the broad level but disagree on **shape**: Holt-Winters carries the annual rhythm into the horizon (a higher early-horizon level that eases to a mid-horizon trough and lifts again), whereas STEPAR — which models seasonality only indirectly through AR lags — decays more smoothly toward the series mean. The gap between them at each step is the seasonal signal STEPAR leaves on the table.

> SAS would normally drive this adequacy check with `OUTRESID` one-step-ahead residuals (`_TYPE_='RESIDUAL'`). Jenner does not yet populate those rows (gap test `400979`), so we compare the two forward forecasts directly instead — a diagnostic that uses only output the engine actually produces.

In [4]:
/* STEPAR horizon, indexed by months-ahead. */
data stepar_h;
    set fc_stepar;
    retain months_ahead 0;
    if _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    keep months_ahead stepar;
run;

/* WINTERS horizon, indexed by months-ahead. */
data winters_h;
    set fc_winters;
    retain months_ahead 0;
    if _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    keep months_ahead winters;
run;

/* Merge the two and show the seasonal gap STEPAR misses. */
data compare;
    merge stepar_h winters_h;
    by months_ahead;
    seasonal_gap = winters - stepar;
run;

proc print data=compare label;
    label months_ahead = 'Months Ahead'
          stepar        = 'STEPAR Forecast'
          winters       = 'Winters Forecast'
          seasonal_gap  = 'Winters - STEPAR';
    format stepar winters seasonal_gap 8.0;
    title 'STEPAR vs Holt-Winters: 12-Month Forecast Comparison';
run;

                                  STEPAR vs Holt-Winters: 12-Month Forecast Comparison                                  

  Obs  Months Ahead  STEPAR Forecast  Winters Forecast  Winters - STEPAR
    1             1             2619              2783               164
    2             2             2537              2897               361
    3             3             2384              2805               421
    4             4             2214              2664               450
    5             5             2089              2629               540
    6             6             2010              2548               539
    7             7             1982              2392               410
    8             8             1995              2240               245
    9             9             2031              2197               166
   10            10             2075              2354               280
   11            11             2115              2423               308
  

## Step 5 — Forecast every line of business at once (BY processing)

Real reserving runs cover several products. With the data sorted by `line_of_business`, a `BY` statement makes `PROC FORECAST` fit an **independent seasonal model for each group** in a single call. Here we synthesize an Auto book (higher base volume) and a Home book (lower base) and forecast both six months ahead. `OUTALL` writes the full set of rows — the `ACTUAL` history and the `FORECAST` horizon — together with the limit columns for every group, and we keep just the `FORECAST` rows for the table below.

In [5]:
data multi_book;
    call streaminit(20240531);
    length line_of_business $6;
    do lob = 1 to 2;
        if lob = 1 then line_of_business = 'Auto';
        else            line_of_business = 'Home';
        do i = 0 to 47;
            date = intnx('month', '01JAN2021'd, i);
            format date monyy7.;
            mi   = mod(i, 12) + 1;
            base = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(base + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi in (12, 1, 2))
                + rand('normal', 0, 70));
            output;
        end;
    end;
    keep line_of_business date claim_count;
run;

proc sort data=multi_book;
    by line_of_business date;
run;

proc forecast data=multi_book
        out=fc_by
        method=winters seasons=12 trend=2
        lead=6 alpha=0.05
        outall;
    by line_of_business;
    id date interval=month;
    var claim_count;
run;

proc print data=fc_by(obs=12) label;
    where _type_ = 'FORECAST';
    title 'Per-Line-of-Business 6-Month Forecasts';
run;

                                  STEPAR vs Holt-Winters: 12-Month Forecast Comparison                                  


BY Group: line_of_business=Auto

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Home

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                         Per-Line-of-Business 6-Month Forecasts                                         

  Obs  LINE_OF_BUSINESS   DATE    _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT  RESIDUAL_CLAIM_COUNT
    1  Auto              23712  FORECAST  2524.596717      2359.095846      2690.097589
    2  Auto              23713  FORECAST  2604.418724      2370.365147        2838.4723
    3  Auto              23714  FORECAST  2576.092313      2289.436395       2862.74823
    4  Auto              23715  FORECAST  2516.029029      2185.027287      2847.030772
    5  Auto 

## Interpreting the results

**What the models tell the reserving team:**

- **STEPAR** reproduces the upward drift and short-run momentum, but its 12-month forecast decays smoothly from about 2,620 (step 1) toward roughly 1,980 mid-horizon before drifting back to about 2,140 — it does not reproduce a sharp winter peak, because it treats seasonality only through autoregressive lags. It is a reasonable quick baseline.
- **WINTERS** with `SEASONS=12` carries the annual rhythm directly through its multiplicative seasonal factors: its forecast is highest in the early horizon (about 2,780 at step 1, about 2,900 at step 2), eases to a trough near step 9 (about 2,200), and lifts again by step 12 (about 2,800). Against the STEPAR baseline it sits **150-660 claims higher** through most of the horizon (see the `Winters - STEPAR` column in Step 4) — that gap is the seasonal signal the autoregressive model leaves out.
- The **95% confidence band** (`L95_*`/`U95_*` columns, controlled by `ALPHA=`) widens as the horizon extends — for the WINTERS model from a width of roughly 410 claims at step 1 to about 1,420 at step 12 — the honest signal that late-horizon estimates carry more uncertainty than near-term ones. Reservists should hold capital against the upper limit, not just the point forecast.
- **BY processing** produces the Auto and Home forecasts in one pass, each with its own seasonal fit. The Auto book forecasts in the roughly 2,510-2,600 range while the Home book sits near 1,870-2,030, so each line keeps its own level and seasonal shape — the pattern the team would extend to every product in the portfolio.

**Bottom line:** for a claim-count series with a clear annual cycle, `METHOD=WINTERS SEASONS=12` is the idiomatic choice; the STEPAR baseline is a useful sanity check, and `OUTFULL`/`OUTLIMIT` plus a step-by-step model comparison let the actuary size the forward reserve with its uncertainty band.

> **Engine fidelity note.** This notebook documents two current Jenner PROC FORECAST limitations (gap test `400979`): the forecast-horizon `ID` is advanced by one unit per step rather than by `INTERVAL=MONTH`, so the printed forecast dates are not the intended 2025 calendar months (we review the horizon by step instead); and `OUTRESID`/`OUTALL` do not yet populate the one-step-ahead `_TYPE_='RESIDUAL'` rows, so residual diagnostics are replaced by a direct two-model comparison. The point forecasts and confidence limits themselves are produced and are what the narrative above is grounded in.